# **1. O que melhora a previsão de cada clube**

O modelo de gradient boosting do notebook 04 usa seis grupos de variáveis. Este notebook responde à pergunta do estudo clube a clube: **retirar qual grupo piora a previsão deste clube?** Cada grupo é retirado por vez, o modelo é treinado de novo na mesma janela expansiva, e o aumento do erro do clube em 2025 e 2026 mede o que o grupo contribui. Uma segunda leitura, com os valores SHAP do modelo completo, mostra quanto cada grupo desloca a previsão de cada clube.

| etapa | método | resultado |
|---|---|---|
| modelo completo | gradient boosting com todos os grupos, três sementes | erro de referência por jogo |
| ablação | o mesmo modelo sem um grupo por vez | aumento do erro por clube e por grupo |
| incerteza | bootstrap pareado sobre os jogos de cada clube | proporção de reamostragens em que o grupo ajuda |
| contribuições | valores SHAP do modelo completo, somados por grupo | tamanho do efeito de cada grupo na previsão |

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import display

### **1.1. Configuração**

O alvo e os hiperparâmetros são lidos do manifesto do notebook 04, para que os dois notebooks avaliem o mesmo modelo. Retirar o grupo de histórico também troca o nível base da média móvel pela mediana do clube nas temporadas de treino; caso contrário, o histórico continuaria entrando pelo alvo.

In [2]:
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"
OUTPUT = PROCESSED / "ablation"
OUTPUT.mkdir(parents=True, exist_ok=True)

manifest = json.loads((PROCESSED / "forecasting" / "run_manifest.json").read_text())
TARGET = manifest["target"]
GBM_PARAMS = manifest["gbm_params"]
TEST_SEASONS = manifest["test_seasons"]
FIRST_SEASON = manifest["first_season"]
SEEDS = [7, 11, 23]
N_BOOT = 2000
MIN_MATCHES = 12
rng = np.random.default_rng(7)

features = pd.read_csv(PROCESSED / "features.csv", parse_dates=["date", "kickoff_local"])
FEATURE_GROUPS = json.loads((PROCESSED / "feature_groups.json").read_text())
ALL_FEATURES = [c for group in FEATURE_GROUPS.values() for c in group]
data = features[features["crowd_status"].eq("regular")].copy().sort_values("kickoff_local").reset_index(drop=True)
data["log_att"] = np.log(data["attendance"])
print(f"alvo: {TARGET} | grupos: {list(FEATURE_GROUPS)} | sementes: {SEEDS}")

alvo: level | grupos: ['history', 'calendar', 'weather', 'form', 'opponent', 'venue'] | sementes: [7, 11, 23]


## **2. Modelo**

As funções repetem as do notebook 04: o fold da janela expansiva, o nível base com a alternativa para promovidos e o ajuste do LightGBM sem a identidade do clube.

In [3]:
def fold(season: int):
    return data[data["season"].between(FIRST_SEASON, season - 1)], data[data["season"].eq(season)]


def base_level(train, frame, use_history=True):
    newcomer_level = train.loc[train["att_history_n"].eq(0), "attendance"].median()
    if not np.isfinite(newcomer_level):
        newcomer_level = train["attendance"].median()
    if use_history:
        return frame["att_ma5"].fillna(newcomer_level).to_numpy()
    return frame["home_id"].map(train.groupby("home_id")["attendance"].median()).fillna(newcomer_level).to_numpy()


def matrix(frame, columns):
    return frame[columns].astype(float)


def fit_predict(train, test, columns, use_history, seed):
    base_train, base_test = base_level(train, train, use_history), base_level(train, test, use_history)
    y = np.log(train["attendance"].to_numpy() / base_train) if TARGET == "ratio" else train["log_att"].to_numpy()
    model = lgb.LGBMRegressor(**{**GBM_PARAMS, "random_state": seed})
    model.fit(matrix(train, columns), y)
    raw = model.predict(matrix(test, columns))
    prediction = np.exp(raw) * base_test if TARGET == "ratio" else np.exp(raw)
    return prediction, model

## **3. Execuções da ablação**

Para cada temporada de teste, o modelo completo e os seis modelos reduzidos são treinados com três sementes, e suas previsões são promediadas, o que remove a maior parte do ruído da amostragem de linhas e colunas.

In [4]:
VARIANTS = {"full": ALL_FEATURES, **{f"without_{g}": [c for c in ALL_FEATURES if c not in cols] for g, cols in FEATURE_GROUPS.items()}}
runs = []
models = {}
for season in TEST_SEASONS:
    train, test = fold(season)
    frame = test[["game_id", "season", "home", "attendance"]].copy()
    for variant, columns in VARIANTS.items():
        use_history = variant != "without_history"
        predictions = []
        for seed in SEEDS:
            prediction, model = fit_predict(train, test, columns, use_history, seed)
            predictions.append(prediction)
            if variant == "full":
                models[(season, seed)] = model
        frame[variant] = np.mean(predictions, axis=0)
    runs.append(frame)
runs = pd.concat(runs, ignore_index=True)
errors = runs[list(VARIANTS)].sub(runs["attendance"], axis=0).abs()
errors[["game_id", "season", "home"]] = runs[["game_id", "season", "home"]]
display(errors[list(VARIANTS)].mean().round(0).to_frame("MAE, 2025–2026"))

,"MAE, 2025–2026"
full,6324.0
without_history,10686.0
without_calendar,6483.0
without_weather,6335.0
without_form,6549.0
without_opponent,6671.0
without_venue,6660.0


## **4. Contribuição de cada grupo**

### **4.1. No conjunto**

O aumento do erro no conjunto dos jogos quando um grupo é retirado, em torcedores e como fração do erro do modelo completo.

In [5]:
GROUPS = list(FEATURE_GROUPS)
full_mae = errors["full"].mean()
overall = pd.DataFrame({
    "mae_without": [errors[f"without_{g}"].mean() for g in GROUPS],
}, index=GROUPS)
overall["increase"] = overall["mae_without"] - full_mae
overall["relative_increase"] = overall["increase"] / full_mae
display(overall.sort_values("increase", ascending=False).round({"mae_without": 0, "increase": 0, "relative_increase": 3}))

,mae_without,increase,relative_increase
history,10686.0,4362.0,0.690
opponent,6671.0,347.0,0.055
venue,6660.0,337.0,0.053
form,6549.0,226.0,0.036
calendar,6483.0,159.0,0.025
weather,6335.0,12.0,0.002


### **4.2. Por clube**

Para cada clube com ao menos doze jogos de teste, o aumento do erro de cada grupo e a proporção de reamostragens bootstrap pareadas em que retirar o grupo piora a previsão do clube. Uma proporção de 0,9 ou mais é lida como um grupo que claramente ajuda aquele clube.

In [6]:
club_rows = []
for club, frame in errors.groupby("home"):
    if len(frame) < MIN_MATCHES:
        continue
    index = rng.integers(0, len(frame), size=(N_BOOT, len(frame)))
    base = frame["full"].to_numpy()
    for group in GROUPS:
        diff = frame[f"without_{group}"].to_numpy() - base
        club_rows.append({
            "club": club, "group": group, "matches": len(frame), "full_mae": base.mean(),
            "increase": diff.mean(), "relative_increase": diff.mean() / base.mean(),
            "share_helps": float((diff[index].mean(axis=1) > 0).mean()),
        })
by_club = pd.DataFrame(club_rows)
increase = by_club.pivot(index="club", columns="group", values="relative_increase")[GROUPS]
confidence = by_club.pivot(index="club", columns="group", values="share_helps")[GROUPS]

exogenous = [g for g in GROUPS if g != "history"]
leader = pd.DataFrame({
    "matches": by_club.groupby("club")["matches"].first(),
    "full_mae": by_club.groupby("club")["full_mae"].first(),
    "most_useful_group": increase[exogenous].idxmax(axis=1),
    "increase": increase[exogenous].max(axis=1),
    "share_helps": [confidence.loc[c, g] for c, g in increase[exogenous].idxmax(axis=1).items()],
    "groups_that_clearly_help": [", ".join(g for g in exogenous if confidence.loc[c, g] >= 0.9) for c in increase.index],
}).sort_values("increase", ascending=False)
display(leader.round({"full_mae": 0, "increase": 3, "share_helps": 2}))
display(leader["most_useful_group"].value_counts().to_frame("clubes em que o grupo mais ajuda"))

,matches,full_mae,most_useful_group,increase,share_helps,groups_that_clearly_help
club,,,,,,
Mirassol,30,1981.0,opponent,0.527,1.00,"calendar, opponent, venue"
Atlético-MG,28,6355.0,form,0.356,1.00,"calendar, form, opponent"
Red Bull Bragantino,27,1367.0,opponent,0.350,0.97,opponent
Palmeiras,27,5855.0,venue,0.244,1.00,"form, venue"
Vasco da Gama,28,4152.0,venue,0.188,0.95,"form, venue"
São Paulo,27,9182.0,venue,0.177,1.00,venue
Sport,19,5081.0,opponent,0.166,0.93,"weather, opponent"
Corinthians,29,5116.0,opponent,0.163,0.98,"opponent, venue"
Ceará,19,8910.0,opponent,0.148,1.00,"form, opponent"


,clubes em que o grupo mais ajuda
most_useful_group,
venue,8
opponent,6
calendar,4
form,2


### **4.3. Contribuições SHAP**

Os valores SHAP do modelo completo decompõem cada previsão (na escala log) nas contribuições das variáveis. Somá-los por grupo e tirar a média do valor absoluto por clube dá a porcentagem típica com que cada grupo desloca a previsão daquele clube.

In [7]:
shap_rows = []
for season in TEST_SEASONS:
    _, test = fold(season)
    contributions = np.mean([models[(season, seed)].predict(matrix(test, ALL_FEATURES), pred_contrib=True) for seed in SEEDS], axis=0)
    contributions = pd.DataFrame(contributions[:, :-1], columns=ALL_FEATURES)
    grouped = pd.DataFrame({g: contributions[cols].sum(axis=1) for g, cols in FEATURE_GROUPS.items()})
    grouped["home"] = test["home"].to_numpy()
    shap_rows.append(grouped)
shap = pd.concat(shap_rows, ignore_index=True)
shap_by_club = shap.groupby("home")[GROUPS].apply(lambda f: (np.exp(f.abs()) - 1).mean())
shap_by_club = shap_by_club.loc[increase.index]
display(shap_by_club.round(3))

,history,calendar,weather,form,opponent,venue
home,,,,,,
Atlético-MG,0.425,0.070,0.018,0.057,0.103,0.030
Bahia,0.738,0.047,0.024,0.060,0.076,0.028
Botafogo,0.160,0.069,0.016,0.085,0.132,0.016
Ceará,0.472,0.060,0.021,0.028,0.089,0.053
Corinthians,0.837,0.048,0.023,0.039,0.088,0.051
Cruzeiro,0.774,0.050,0.018,0.087,0.074,0.028
Flamengo,1.531,0.042,0.011,0.107,0.063,0.031
Fluminense,0.422,0.055,0.016,0.077,0.113,0.020
Fortaleza,0.377,0.043,0.023,0.079,0.100,0.049


## **5. Exportação e auditoria**

| arquivo | conteúdo |
|---|---|
| `data/processed/ablation/match_errors.csv` | erro absoluto do modelo completo e dos reduzidos por jogo de teste |
| `data/processed/ablation/overall.csv` | aumento do erro no conjunto por grupo |
| `data/processed/ablation/by_club.csv` | aumento do erro e proporção bootstrap por clube e grupo |
| `data/processed/ablation/club_summary.csv` | grupo exógeno mais útil de cada clube |
| `data/processed/ablation/shap_by_club.csv` | efeito SHAP absoluto médio por clube e grupo |

In [8]:
errors.to_csv(OUTPUT / "match_errors.csv", index=False)
overall.to_csv(OUTPUT / "overall.csv")
by_club.to_csv(OUTPUT / "by_club.csv", index=False)
leader.to_csv(OUTPUT / "club_summary.csv")
shap_by_club.to_csv(OUTPUT / "shap_by_club.csv")

reference = pd.read_csv(PROCESSED / "forecasting" / "predictions.csv")
reference_mae = (reference["attendance"] - reference["pred_gbm"]).abs().mean()
assert abs(full_mae - reference_mae) / reference_mae < 0.05, f"MAE do modelo completo {full_mae:.0f} difere do notebook 04 ({reference_mae:.0f})"
assert set(errors["season"]) == set(TEST_SEASONS), "A ablação usou uma temporada fora da janela de teste"
assert by_club["share_helps"].between(0, 1).all(), "Proporção bootstrap inválida"